# COMP6713 — Notebook 5: Full Evaluation & Comparison

This notebook presents the **end-to-end evaluation** of all three models:

| Model | Description |
|-------|-------------|
| **TF-IDF Baseline** | Keyword-matching retrieval, no learning |
| **Fine-tuned PubMedBERT** | Extractive QA — predicts an answer span within the context |
| **RAG + Qwen-14B** | Retrieval-Augmented Generation — retrieves evidence, then generates a free-text answer |

### Metrics
- **Exact Match (EM)**: 1 if the normalised prediction exactly equals the gold answer
- **Token F1**: token-level overlap (same as SQuAD leaderboard)
- **BERTScore F1**: semantic similarity via contextual embeddings (RoBERTa-base)
- **ROUGE-L**: longest common subsequence overlap

All models are evaluated on the same **held-out test set** (20% of the combined 20,667-record corpus).

In [ ]:
%matplotlib inline
import sys, warnings, os, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Load Pre-computed Results

In [ ]:
# Load results saved by run_eval.py
results_path = '../reports/evaluation_results_default.json'

with open(results_path) as f:
    results = json.load(f)

# Display as a DataFrame
df_results = pd.DataFrame(results).T.rename(columns={
    'EM': 'Exact Match',
    'F1': 'Token F1',
    'BERTScore': 'BERTScore F1',
    'ROUGE-L': 'ROUGE-L',
})

print("Evaluation results (n=100 test samples)")
print(df_results.round(4).to_string())

## 2. Visual Comparison — All Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models   = list(results.keys())
metrics  = ['EM', 'F1', 'BERTScore', 'ROUGE-L']
metric_labels = ['Exact Match', 'Token F1', 'BERTScore F1', 'ROUGE-L']
colors   = ['steelblue', 'darkorange', 'seagreen']

x = np.arange(len(metrics))
width = 0.22

# — Left plot: all 4 metrics side-by-side —
for i, (model, color) in enumerate(zip(models, colors)):
    vals = [results[model][m] for m in metrics]
    axes[0].bar(x + i*width, vals, width, label=model, color=color, alpha=0.85)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metric_labels, fontsize=9)
axes[0].set_ylabel('Score')
axes[0].set_title('Model Comparison — All Metrics')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.05)
axes[0].axhline(y=0, color='black', linewidth=0.5)

# — Right plot: radar / polar chart —
angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

ax2 = fig.add_subplot(122, polar=True)
for model, color in zip(models, colors):
    vals = [results[model][m] for m in metrics]
    vals += vals[:1]
    ax2.plot(angles, vals, 'o-', color=color, linewidth=1.5, label=model)
    ax2.fill(angles, vals, color=color, alpha=0.1)

ax2.set_thetagrids(np.degrees(angles[:-1]), metric_labels, fontsize=8)
ax2.set_ylim(0, 1)
ax2.set_title('Radar Chart — Model Profiles', pad=15)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/eval_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved → reports/figures/eval_comparison.png")

## 3. Per-metric Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, metric, label in zip(axes, metrics, metric_labels):
    vals  = [results[m][metric] for m in models]
    bars  = ax.bar(models, vals, color=colors, alpha=0.85, edgecolor='white')
    ax.set_title(label, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    # Annotate bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Per-metric Comparison (n=100 test samples)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/eval_per_metric.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Results Discussion

### Key Findings

**Exact Match (EM)**  
Fine-tuned PubMedBERT achieves EM = 0.51, far above the Baseline (0.00) and RAG+Qwen (0.02).  
This is expected: EM rewards exact string matching, which favours extractive models that copy  
answer spans directly from the source text.

**Token F1**  
BERT again leads (F1 = 0.555), followed by RAG (0.147), then Baseline (0.026).  
The large gap between BERT and RAG on F1/EM is a measurement artefact — F1 and EM  
are designed for *extractive* QA where the answer is a verbatim substring of the context.  
RAG generates *paraphrased* answers that carry the right meaning but differ in wording.

**BERTScore F1** *(semantic similarity)*  
BERTScore closes the gap significantly:  
- BERT:     0.890  
- RAG:      0.835  
- Baseline: 0.784  

All three scores are in a narrower range here because BERTScore captures semantic  
equivalence rather than exact tokens. This metric better reflects RAG's true quality.

**ROUGE-L**  
Consistent with F1 — BERT leads (0.564), RAG is middle (0.150), Baseline is last (0.028).

### Why does the Baseline have high BERTScore despite near-zero EM/F1?
The TF-IDF baseline retrieves contextually related text but not the exact answer span.  
Its output is semantically plausible (hence BERTScore ≈ 0.78) but rarely a verbatim match.

## 5. Error Analysis

In [ ]:
from medqa.data.loader import load_all
from medqa.data.preprocessor import split_dataset, normalise_answer
from medqa.models.baseline import TFIDFBaseline
from medqa.models.bert_qa import PubMedBERTQA
from medqa.evaluation.metrics import exact_match, token_f1

all_records = load_all()
_, test = split_dataset(all_records)
sample = test[:100]

questions = [r['question'] for r in sample]
contexts  = [r['context']  for r in sample]
golds     = [r['answer']   for r in sample]
sources   = [r['source']   for r in sample]

In [ ]:
# Get BERT predictions (model must be fine-tuned/loaded)
bert = PubMedBERTQA()
bert_loaded = bert.load()

if bert_loaded:
    bert_preds_raw = bert.batch_predict(questions, contexts)
    bert_preds = [p['predicted_answer'] for p in bert_preds_raw]
    bert_scores = [p['score'] for p in bert_preds_raw]
else:
    print("BERT model not loaded — showing example error analysis only")
    bert_preds  = [""] * len(questions)
    bert_scores = [0.0] * len(questions)

In [ ]:
# Build per-sample analysis dataframe
records_df = pd.DataFrame({
    'question':   questions,
    'gold':       golds,
    'bert_pred':  bert_preds,
    'source':     sources,
    'bert_em':    [exact_match(p, g) for p, g in zip(bert_preds, golds)],
    'bert_f1':    [token_f1(p, g)    for p, g in zip(bert_preds, golds)],
    'confidence': bert_scores,
})

print(f"BERT EM:  {records_df['bert_em'].mean():.3f}")
print(f"BERT F1:  {records_df['bert_f1'].mean():.3f}")
print()
print(f"Correct (EM=1): {int(records_df['bert_em'].sum())} / {len(records_df)}")
print(f"Wrong   (EM=0): {int((records_df['bert_em']==0).sum())} / {len(records_df)}")

In [ ]:
# EM by dataset source
print("Exact Match breakdown by dataset:")
print(records_df.groupby('source')[['bert_em','bert_f1']].mean().round(3))

In [ ]:
# Plot EM and F1 distribution by source
source_stats = records_df.groupby('source')[['bert_em','bert_f1']].mean()

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(source_stats))
ax.bar(x - 0.2, source_stats['bert_em'], 0.35, label='Exact Match', color='steelblue', alpha=0.85)
ax.bar(x + 0.2, source_stats['bert_f1'], 0.35, label='Token F1',    color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(source_stats.index, fontsize=9)
ax.set_ylabel('Score'); ax.set_ylim(0, 1)
ax.set_title('PubMedBERT Performance by Dataset Source')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/bert_by_source.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Show failure cases — low F1 but high confidence
failures = records_df[records_df['bert_f1'] < 0.3].sort_values('confidence', ascending=False)

print(f"Failure cases (F1 < 0.3): {len(failures)}")
print()
for _, row in failures.head(5).iterrows():
    print(f"Source   : {row['source']}")
    print(f"Question : {row['question'][:100]}")
    print(f"Gold     : {row['gold'][:100]}")
    print(f"Predicted: {row['bert_pred'][:100]}")
    print(f"F1={row['bert_f1']:.3f}  Confidence={row['confidence']:.3f}")
    print("-" * 60)

In [ ]:
# Confidence calibration: does the model know when it's wrong?
if bert_loaded:
    bins = np.linspace(0, 1, 6)
    records_df['conf_bin'] = pd.cut(records_df['confidence'], bins=bins)
    calib = records_df.groupby('conf_bin')['bert_em'].mean()

    fig, ax = plt.subplots(figsize=(7, 4))
    calib.plot(kind='bar', ax=ax, color='teal', alpha=0.8, rot=30)
    ax.axline((0, 0), slope=1/5, color='red', linestyle='--', label='Perfect calibration')
    ax.set_title('Confidence Calibration (Exact Match vs Confidence Bin)')
    ax.set_xlabel('Confidence score bin'); ax.set_ylabel('Mean EM in bin')
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('../reports/figures/confidence_calibration.png', dpi=120, bbox_inches='tight')
    plt.show()

## 6. Answer Length Analysis

In [ ]:
# How does prediction length compare to gold length?
records_df['gold_len']       = records_df['gold'].str.len()
records_df['bert_pred_len']  = records_df['bert_pred'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(records_df['gold_len'], records_df['bert_pred_len'],
                alpha=0.4, s=20, color='steelblue')
axes[0].plot([0, records_df['gold_len'].max()], [0, records_df['gold_len'].max()],
             'r--', linewidth=1, label='y=x (perfect)')
axes[0].set_xlabel('Gold answer length (chars)')
axes[0].set_ylabel('Predicted answer length (chars)')
axes[0].set_title('PubMedBERT: Predicted vs Gold answer length')
axes[0].legend()

axes[1].hist(records_df['gold_len'],      bins=20, alpha=0.6, label='Gold',      color='darkorange')
axes[1].hist(records_df['bert_pred_len'], bins=20, alpha=0.6, label='Predicted', color='steelblue')
axes[1].set_xlabel('Answer length (chars)'); axes[1].set_ylabel('Count')
axes[1].set_title('Answer length distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/answer_length.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Mean gold length : {records_df['gold_len'].mean():.0f} chars")
print(f"Mean pred length : {records_df['bert_pred_len'].mean():.0f} chars")

## 7. Summary Table

In [ ]:
print("\n" + "="*68)
print(f"  {'Model':<22} {'EM':>6} {'F1':>6} {'BERTScore':>10} {'ROUGE-L':>8}")
print("="*68)
for model, r in results.items():
    print(f"  {model:<22} {r['EM']:>6.3f} {r['F1']:>6.3f} {r['BERTScore']:>10.3f} {r['ROUGE-L']:>8.3f}")
print("="*68)
print()
print("Notes:")
print("  * Evaluated on 100 randomly sampled test records.")
print("  * EM and F1 favour extractive answers (BERT); BERTScore better reflects")
print("    the true quality of generative answers (RAG + Qwen).")
print("  * RAG's low EM/F1 is expected — it generates paraphrased rather than verbatim answers.")

## 8. Conclusions

### Performance Summary
Fine-tuned **PubMedBERT** achieves the highest scores on all token-overlap metrics  
(EM = 0.51, F1 = 0.56, ROUGE-L = 0.56) because it performs extractive QA —  
copying exact spans from the context that match the ground-truth annotation verbatim.

The **RAG + Qwen-14B** pipeline achieves much lower EM/F1 (0.02 / 0.15) but a  
competitive BERTScore of 0.835, indicating its answers are *semantically* correct  
even when the wording differs. This is a well-known limitation of lexical metrics  
when applied to generative models.

The **TF-IDF Baseline** scores near zero on EM/F1 but shows a surprisingly high  
BERTScore (0.784), reflecting that retrieved passages are topically relevant even  
without answer extraction.

### Limitations & Future Work
- EM/F1 are suboptimal for generative QA; a clinical NLI-based metric would be fairer.
- BERT is limited to extractive answers; it cannot synthesise information across passages.
- The RAG pipeline could be improved by fine-tuning the LLM on medical QA.
- Error analysis reveals BERT struggles most on MedQA-USMLE (multiple-choice),  
  where the "answer" is an option string rather than a context span.